# Data Cleaning & Preparation - Credit Card Fraud Detection

This notebook implements the full cleaning pipeline to make the data **training-ready** for a fraud detection model.

## Pipeline Overview
1. **Load** raw data from `dataset/fraudTrain.csv`
2. **Drop** high-cardinality identifier columns
3. **Feature engineering**: time features, age, merchant distance
4. **Encode** categoricals: label encoding (category, state), frequency encoding (job, merchant)
5. **Scale** numeric features
6. **Downsample** 75% of normal transactions
7. **Apply SMOTE** to rebalance fraud:normal to 1:4
8. **Save** clean, training-ready dataset

## Key Decisions
- **Downsampling**: Remove 75% of non-fraudulent transactions (1,289,169 → 322,292)
- **SMOTE ratio**: 1:4 fraud:normal → 80,573 fraud samples
- **Encodings**: Label encode `category` & `state`, frequency encode `job` & `merchant`
- **Coordinates**: `lat`, `long`, `merch_lat`, `merch_long` kept raw
- **Random state**: 42 (reproducible)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from math import radians, sin, cos, sqrt, asin
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded successfully')

Libraries loaded successfully


## 1. Load Data

In [2]:
# Load the dataset
df = pd.read_csv('dataset/fraudTrain.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'\nClass distribution:')
print(df['is_fraud'].value_counts())
print(f'\nFraud rate: {df["is_fraud"].mean() * 100:.4f}%')

Loaded: 1,296,675 rows, 23 columns

Class distribution:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Fraud rate: 0.5789%


## 2. Drop High-Cardinality Identifier Columns

In [3]:
# Columns to drop: unique identifiers or non-predictive metadata
drop_cols = ['Unnamed: 0', 'trans_num', 'cc_num', 'first', 'last', 'street', 'city', 'zip']
df = df.drop(columns=drop_cols)
print(f'Dropped columns: {drop_cols}')
print(f'Remaining shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'\nRemaining columns: {df.columns.tolist()}')

Dropped columns: ['Unnamed: 0', 'trans_num', 'cc_num', 'first', 'last', 'street', 'city', 'zip']
Remaining shape: 1,296,675 rows, 15 columns

Remaining columns: ['trans_date_trans_time', 'merchant', 'category', 'amt', 'gender', 'state', 'lat', 'long', 'city_pop', 'job', 'dob', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


## 3. Feature Engineering

In [4]:
# Parse datetime columns
df['trans_date'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

# Extract temporal features
df['hour'] = df['trans_date'].dt.hour
df['day_of_week'] = df['trans_date'].dt.dayofweek
df['month'] = df['trans_date'].dt.month

# Compute age at transaction time
df['age'] = (df['trans_date'] - df['dob']).dt.days / 365.25

# Drop raw datetime columns
df = df.drop(columns=['trans_date_trans_time', 'dob', 'trans_date'])

print('Feature engineering complete:')
print(f'  - hour, day_of_week, month extracted')
print(f'  - age computed from DOB')
print(f'  - Raw datetime columns dropped')
print(f'\nShape: {df.shape[0]:,} rows, {df.shape[1]} columns')

Feature engineering complete:
  - hour, day_of_week, month extracted
  - age computed from DOB
  - Raw datetime columns dropped

Shape: 1,296,675 rows, 17 columns


In [5]:
# Haversine distance between cardholder and merchant
def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in km between two points"""
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * R * asin(min(1, sqrt(a)))

df['merchant_distance'] = df.apply(
    lambda row: haversine(row['lat'], row['long'], row['merch_lat'], row['merch_long']),
    axis=1
)

print(f'Merchant distance computed')
print(f'  - Mean distance: {df["merchant_distance"].mean():.2f} km')
print(f'  - Max distance: {df["merchant_distance"].max():.2f} km')

Merchant distance computed
  - Mean distance: 76.11 km
  - Max distance: 152.12 km


## 4. Encode Categorical Features

In [6]:
# Label encode: category, state
label_encoders = {}
for col in ['category', 'state']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f'Label encoded: {col} ({le.classes_.size} classes)')

# Binary encode: gender (F=0, M=1 manually to be explicit)
df['gender'] = df['gender'].map({'F': 0, 'M': 1})
print(f'\nGender binary encoded: {{0: Female, 1: Male}}')

Label encoded: category (14 classes)
Label encoded: state (51 classes)



Gender binary encoded: {0: Female, 1: Male}


In [7]:
# Frequency encode: job and merchant
def frequency_encode(df, column):
    """Replace values with their frequency count in the dataset"""
    freq_map = df[column].value_counts()
    df[f'{column}_freq'] = df[column].map(freq_map)
    return df

for col in ['job', 'merchant']:
    df = frequency_encode(df, col)
    print(f'Frequency encoded: {col}')

# Drop original high-cardinality columns after encoding
df = df.drop(columns=['job', 'merchant'])
print(f'\nDropped original job & merchant columns')
print(f'Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')

Frequency encoded: job


Frequency encoded: merchant

Dropped original job & merchant columns
Shape: 1,296,675 rows, 18 columns


## 5. Scale Numeric Features

In [8]:
# Columns to scale (excluding target)
scale_cols = ['amt', 'hour', 'day_of_week', 'month', 'age', 'city_pop',
              'unix_time', 'merchant_distance', 'lat', 'long', 'merch_lat', 'merch_long']

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print(f'Scaled columns: {scale_cols}')
print(f'\nScaled stats for amt: mean={df["amt"].mean():.6f}, std={df["amt"].std():.6f}')

Scaled columns: ['amt', 'hour', 'day_of_week', 'month', 'age', 'city_pop', 'unix_time', 'merchant_distance', 'lat', 'long', 'merch_lat', 'merch_long']

Scaled stats for amt: mean=0.000000, std=1.000000


## 6. Downsample Non-Fraudulent Transactions (Remove 75%)

In [9]:
# Separate features and target BEFORE sampling
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

# Downsample normal transactions (keep only 25%)
normal_mask = (y == 0)
fraud_mask = (y == 1)

# Get all fraud samples
X_fraud = X[fraud_mask]
y_fraud = y[fraud_mask]

# Sample 25% of normal transactions
X_normal = X[normal_mask]
y_normal = y[normal_mask]

# Calculate sample size (keep 25% = remove 75%)
sample_size = int(len(X_normal) * 0.25)
X_normal_sample = X_normal.sample(n=sample_size, random_state=RANDOM_STATE)
y_normal_sample = y_normal.loc[X_normal_sample.index]

print(f'Original normal transactions: {len(X_normal):,}')
print(f'After removing 75%: {len(X_normal_sample):,}')
print(f'Fraud transactions kept: {len(X_fraud):,}')
print(f'\nCombined before SMOTE: {len(X_normal_sample) + len(X_fraud):,} rows')

# Recombine
X_sampled = pd.concat([X_normal_sample, X_fraud])
y_sampled = pd.concat([y_normal_sample, y_fraud])

print(f'\nClass distribution after downsampling:')
print(y_sampled.value_counts())

Original normal transactions: 1,289,169
After removing 75%: 322,292
Fraud transactions kept: 7,506

Combined before SMOTE: 329,798 rows

Class distribution after downsampling:
is_fraud
0    322292
1      7506
Name: count, dtype: int64


## 7. Apply SMOTE (1:4 Fraud:Normal Ratio)

In [10]:
# SMOTE with 1:4 fraud:normal ratio
# If normal = N, fraud target = N/4
n_normal = (y_sampled == 0).sum()
n_fraud_target = int(n_normal / 4)

print(f'Normal samples: {n_normal:,}')
print(f'Target fraud samples (1:4 ratio): {n_fraud_target:,}')

smote = SMOTE(
    sampling_strategy={1: n_fraud_target},
    random_state=RANDOM_STATE,
    k_neighbors=5
)

X_resampled, y_resampled = smote.fit_resample(X_sampled, y_sampled)

print(f'\nAfter SMOTE:')
print(f'  - Shape: {X_resampled.shape[0]:,} rows, {X_resampled.shape[1]} features')
print(f'  - Normal: {(y_resampled == 0).sum():,}')
print(f'  - Fraud: {(y_resampled == 1).sum():,}')
print(f'  - Ratio: 1:{((y_resampled == 0).sum() / (y_resampled == 1).sum()):.2f}')

Normal samples: 322,292
Target fraud samples (1:4 ratio): 80,573



After SMOTE:
  - Shape: 402,865 rows, 17 features
  - Normal: 322,292
  - Fraud: 80,573
  - Ratio: 1:4.00


## 8. Save Cleaned Dataset

In [11]:
# Recombine into final dataframe
df_clean = X_resampled.copy()
df_clean['is_fraud'] = y_resampled.values

# Save to CSV
output_path = 'dataset/cleaned_creditcard.csv'
df_clean.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')
print(f'Final dataset: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns')

Saved to: dataset/cleaned_creditcard.csv
Final dataset: 402,865 rows, 18 columns


## 9. Final Validation

In [12]:
# Reload and verify
df_check = pd.read_csv('dataset/cleaned_creditcard.csv')

print('=== FINAL DATASET VALIDATION ===')
print(f'Shape: {df_check.shape[0]:,} rows, {df_check.shape[1]} columns')
print(f'\nClass distribution:')
print(df_check['is_fraud'].value_counts())
print(f'  - Fraud ratio: {df_check["is_fraud"].mean() * 100:.2f}%')

# Check for missing values
print(f'\nMissing values: {df_check.isnull().sum().sum()}')

# Check data types
print(f'\nData types:')
print(df_check.dtypes)

# Check for infinities
print(f'\nInfinite values: {np.isinf(df_check.select_dtypes(include=[np.number])).sum().sum()}')

print('\n✅ Dataset is TRAINING-READY!')

=== FINAL DATASET VALIDATION ===
Shape: 402,865 rows, 18 columns

Class distribution:
is_fraud
0    322292
1     80573
Name: count, dtype: int64
  - Fraud ratio: 20.00%

Missing values: 0

Data types:
category               int64
amt                  float64
gender                 int64
state                  int64
lat                  float64
long                 float64
city_pop             float64
unix_time            float64
merch_lat            float64
merch_long           float64
hour                 float64
day_of_week          float64
month                float64
age                  float64
merchant_distance    float64
job_freq               int64
merchant_freq          int64
is_fraud               int64
dtype: object

Infinite values: 0

✅ Dataset is TRAINING-READY!


In [13]:
# Final summary of what was done
print('''
=== CLEANING PIPELINE SUMMARY ===
1. Dropped identifiers: Unnamed: 0, trans_num, cc_num, first, last, street, city, zip
2. Feature engineering: hour, day_of_week, month, age, merchant_distance
3. Encodings: 
   - Label: category, state
   - Binary: gender (F=0, M=1)
   - Frequency: job, merchant
4. Scaling: StandardScaler on all numeric features
5. Downsampled 75% of normal transactions: 1,289,169 → 322,292
6. SMOTE applied: 7,506 → 80,573 fraud samples (1:4 ratio)
7. Output: dataset/cleaned_creditcard.csv

Final: 402,865 rows, 18 features (including target)
''')


=== CLEANING PIPELINE SUMMARY ===
1. Dropped identifiers: Unnamed: 0, trans_num, cc_num, first, last, street, city, zip
2. Feature engineering: hour, day_of_week, month, age, merchant_distance
3. Encodings: 
   - Label: category, state
   - Binary: gender (F=0, M=1)
   - Frequency: job, merchant
4. Scaling: StandardScaler on all numeric features
5. Downsampled 75% of normal transactions: 1,289,169 → 322,292
6. SMOTE applied: 7,506 → 80,573 fraud samples (1:4 ratio)
7. Output: dataset/cleaned_creditcard.csv

Final: 402,865 rows, 18 features (including target)

